# Customer Churn Analysis
Step-by-step notebook version of the project.

In [ ]:
import sys; sys.path.append('../src')
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from utils import load_clean, DATA
sns.set_theme(style='whitegrid')

## 1. Load raw data

In [ ]:
raw = pd.read_csv(DATA)
print(raw.shape)
raw.head()

In [ ]:
raw.info()

## 2. Clean data
TotalCharges is text; blanks belong to tenure = 0 customers.

In [ ]:
df = load_clean()
print(df.shape, df.isna().sum().sum())
df.head()

## 3. EDA

In [ ]:
print(f'Churn rate: {df.Churn.mean()*100:.1f}%')
df.groupby('Contract').Churn.mean().mul(100).round(1)

In [ ]:
df.groupby('tenure_group').Churn.mean().mul(100).plot.bar(color='#2E86AB')
plt.ylabel('Churn %'); plt.title('Churn by tenure group'); plt.show()

In [ ]:
sns.heatmap(df.select_dtypes('number').corr(), annot=True, cmap='coolwarm'); plt.show()

## 4. Modeling
Run `python ../src/model.py` for the full comparison, or build your own below.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

X, y = df.drop(columns='Churn'), df.Churn
num = X.select_dtypes('number').columns
cat = [c for c in X.columns if c not in num]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
pre = ColumnTransformer([('n', StandardScaler(), num), ('c', OneHotEncoder(handle_unknown='ignore'), cat)])
pipe = Pipeline([('pre', pre), ('m', LogisticRegression(max_iter=1000, class_weight='balanced'))]).fit(Xtr, ytr)
print(classification_report(yte, pipe.predict(Xte)))
print('AUC', roc_auc_score(yte, pipe.predict_proba(Xte)[:,1]))

## 5. Conclusions
(Write your insights and recommendations here.)